# Cvičení 08 - Matplotlib a NumPy

Cíle cvičení:

* nakreslit graf s popisky, legendou a více řadami
* přepsat výpočet se seznamy na výpočet s polem NumPy
* najít maximum a jeho polohu pomocí `max` a `argmax`
* použít masku k výběru, k přiřazení i do grafu
* porovnat dvě varianty simulací Monte Carlo
* vyzkoušet interaktivní graf s posuvníky
* pracovat s obrázkem jako s 2D polem

Na konci najdete slovníček pojmů.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Úloha 1 - Pracovní diagram betonu

Beton v tlaku se v normě popisuje diagramem parabola-obdélník. Pro beton
C30/37 je návrhová pevnost `f_cd = 30 / 1.5 = 20 MPa` a napětí závisí na
přetvoření `eps` (v promile) takto:

* pro `0 <= eps <= 2`: `sigma = f_cd * (1 - (1 - eps / 2) ** 2)`
* pro `2 < eps <= 3.5`: `sigma = f_cd`

**Část A - seznamy.**

1. vytvořte seznam `eps_seznam` s hodnotami 0, 0.1, 0.2, ..., 3.5
   (36 hodnot, pomůže `range` a list comprehension)
2. cyklem s `if` spočítejte seznam `sigma_seznam`
3. nakreslete graf s popisky os a titulkem

**Část B - NumPy.**

1. vytvořte `eps` funkcí `np.linspace` se stejnými 36 body
2. spočítejte `sigma` vzorcem pro parabolu na celém poli najednou
3. maskou přiřaďte `f_cd` tam, kde `eps > 2`
4. nakreslete obě řešení do jednoho grafu: seznamy čárou, pole jen body
   (`marker="o"`, `linestyle=""`) a přidejte legendu
5. ověřte, že se obě řešení shodují: největší rozdíl
   `np.abs(np.array(sigma_seznam) - sigma).max()` musí být prakticky 0

**Kontrola:** pro `eps = 1` vyjde `sigma = 15 MPa`.

In [ ]:
f_cd = 30 / 1.5     # MPa

# TODO: A1) eps_seznam
# TODO: A2) sigma_seznam cyklem s if
# TODO: A3) graf

In [ ]:
# TODO: B1) eps = np.linspace(...)
# TODO: B2) sigma vzorcem pro parabolu
# TODO: B3) sigma[...] = f_cd
# TODO: B4) obe reseni do jednoho grafu s legendou
# TODO: B5) nejvetsi rozdil

## Úloha 2 - Prostý nosník

Prostý nosník o rozpětí `L` je zatížen spojitým zatížením `q`. Pro moment
a posouvající sílu platí:

* `M(x) = q * x * (L - x) / 2`
* `V(x) = q * (L / 2 - x)`

1. napište funkci `vnitrni_sily(x, L, q)`, která pro pole `x` vrátí dvě pole,
   `M` a `V`
2. pro `L = 6 m` a `q = 12 kN/m` spočítejte `M` a `V` v 201 bodech
3. nakreslete dva grafy nad sebou: `fig, (ax1, ax2) = plt.subplots(2, 1,
   sharex=True)`; nahoře `M`, dole `V`, oba s popisky os a vodorovnou čarou
   v nule (`ax.axhline(0, color="black")`)
4. najděte největší moment a jeho polohu (`max`, `argmax`) a označte ho
   v grafu šipkou s popiskem:

   ```python
   ax1.annotate(f"M_max = {M[i]:.1f} kNm", xy=(x[i], M[i]),
                xytext=(x[i] + 0.5, M[i] - 15), arrowprops={"arrowstyle": "->"})
   ```

5. přidejte vlastní tíhu trámu 300 x 600 mm, `g = 25 * 0.3 * 0.6` kN/m;
   spočítejte moment od `g` stejnou funkcí, sečtěte ho s momentem od `q`
   a nakreslete do horního grafu všechny tři křivky s legendou

**Kontrola:** `M_max = 54 kNm` v `x = 3 m`, `V` na podporách `+-36 kN`. S vlastní
tíhou `M_max = 74.25 kNm`.

In [ ]:
L = 6.0       # m
q = 12.0      # kN/m

# TODO: 1) def vnitrni_sily(x, L, q)
# TODO: 2) x, M, V
# TODO: 3) dva grafy nad sebou
# TODO: 4) maximum a annotate
# TODO: 5) vlastni tiha a soucet

## Úloha 3 - Kde nosník nevyhoví

Průřez nosníku z úlohy 2 má únosnost `M_Rd = 40 kNm`. Počítejte jen se
zatížením `q = 12 kN/m`, bez vlastní tíhy.

1. vytvořte masku `nevyhovi` pro body, kde `M > M_Rd`
2. vypište, kolik bodů nevyhovuje, a od kterého do kterého `x`
3. délka nevyhovujícího úseku je přibližně počet bodů krát vzdálenost dvou
   sousedních bodů, `x[1] - x[0]`; vypište ji
4. nakreslete `M`, vodorovnou čáru `M_Rd` a vybarvěte nevyhovující oblast
   (`fill_between` s `where`)
5. cyklem projděte zatížení `[8, 9, 10, 12, 15]` kN/m a pro každé vypište
   délku nevyhovujícího úseku; od jakého `q` nosník nevyhoví?

**Kontrola:** pro `q = 12` nevyhovuje úsek od 1.50 do 4.50 m, délka vyjde
asi 3.0 m. Přesné řešení rovnice je 1.47 až 4.53 m, rozdíl dělá vzdálenost
bodů. Nosník nevyhoví od `q = 9 kN/m`.

In [ ]:
M_Rd = 40.0     # kNm

# TODO: 1) maska nevyhovi
# TODO: 2) pocet bodu, od-do
# TODO: 3) delka useku
# TODO: 4) graf s fill_between
# TODO: 5) cyklus pres zatizeni

## Úloha 4 - Monte Carlo: dva dodavatelé betonu

Stavba objednává beton C25/30, požadovaná charakteristická pevnost je
`f_ck = 25 MPa`. Nabízejí se dva dodavatelé:

| Dodavatel | průměrná pevnost | směrodatná odchylka |
|---|---|---|
| A | 33 MPa | 5 MPa |
| B | 31 MPa | 3 MPa |

Dodavatel A má vyšší průměr. Je proto lepší?

1. vytvořte generátor `rng = np.random.default_rng(seed=1)`
2. pro každého dodavatele nasimulujte 10 000 vzorků (`rng.normal`)
3. pro každého spočítejte 5% kvantil (`np.percentile`) a podíl vzorků pod
   25 MPa
4. nakreslete oba histogramy do jednoho grafu, poloprůhledně (`alpha=0.5`,
   `bins=60`), se svislou čarou v 25 MPa a s legendou
5. kterého dodavatele byste vybrali a proč?

**Kontrola:** pod 25 MPa je u dodavatele A asi 5.4 % vzorků, u dodavatele B
asi 2.2 %.

In [ ]:
# TODO: 1) generator
# TODO: 2) vzorky A a B
# TODO: 3) kvantily a podil pod 25 MPa
# TODO: 4) dva histogramy v jednom grafu
# TODO: 5) ktereho dodavatele a proc?

## Úloha 5 - Interaktivní nosník

Funkce `interact` z balíčku `ipywidgets` vytvoří ke každému parametru
funkce posuvník. Při každém posunutí funkci zavolá znovu a graf se
překreslí.

Zápis `L=(2.0, 12.0, 0.5)` znamená posuvník od 2 do 12 s krokem 0.5.
Úloha používá funkci `vnitrni_sily` z úlohy 2, takže buňku s ní musíte mít
spuštěnou.

1. doplňte funkci `nakresli` tak, aby nakreslila moment z úlohy 2, čáru
   `M_Rd = 40 kNm` a vybarvila oblast, kde nosník nevyhoví
2. nastavte pevný rozsah osy y, `ax.set_ylim(0, 150)`, aby graf při posouvání
   neposkakoval
3. posuvníky najděte největší rozpětí, při kterém nosník se zatížením
   `q = 12 kN/m` ještě vyhoví

In [ ]:
from ipywidgets import interact


def nakresli(L=6.0, q=12.0):
    x = np.linspace(0, L, 201)
    M, V = vnitrni_sily(x, L, q)
    fig, ax = plt.subplots()
    # TODO: 1) M, cara M_Rd a vybarvena oblast
    # TODO: 2) ax.set_ylim(0, 150)
    plt.show()


interact(nakresli, L=(2.0, 12.0, 0.5), q=(1.0, 30.0, 1.0))

**Kontrola:** s krokem 0.5 m vyhoví nosník nejvýše při rozpětí 5 m.

## Úloha 6 - Obrázek je pole čísel

Černobílá fotka je 2D pole: každé číslo je jas jednoho pixelu, 0 je černá,
1 bílá. Tak s fotkami pracují programy, které na snímcích z dronu hledají
trhliny v betonu.

Buňka níže vyrobí "fotku" betonového povrchu s jednou trhlinou. Jeden pixel
odpovídá 0.5 mm.

1. obrázek nakreslete: `ax.imshow(povrch, cmap="gray", vmin=0, vmax=1)`
   a vypište jeho rozměry (`shape`)
2. nakreslete jen výřez s trhlinou: řádky 20 až 80, sloupce 20 až 100
3. nakreslete obrázek převrácený vzhůru nohama (výřez s krokem `-1`)
4. vytvořte masku `trhlina` pro pixely tmavší než 0.35 a nakreslete ji
   pomocí `imshow`
5. spočítejte, kolik pixelů patří trhlině, a z toho plochu trhliny v mm2
   (jeden pixel má 0.5 x 0.5 mm)

**Kontrola:** trhlina má 280 pixelů, tedy 70 mm2.

In [ ]:
# Vyroba obrazku, nemusite ji menit
rng = np.random.default_rng(seed=7)
povrch = rng.normal(0.65, 0.05, size=(120, 200))    # sedy povrch betonu se sumem

radek = 40
for sloupec in range(30, 170):                     # trhlina: v kazdem sloupci dva tmave pixely
    radek = radek + rng.integers(-1, 2)            # o radek nahoru, dolu nebo rovne
    povrch[radek, sloupec] = 0.15
    povrch[radek + 1, sloupec] = 0.20

In [ ]:
# TODO: 1) imshow a shape
# TODO: 2) vyrez s trhlinou
# TODO: 3) prevraceny obrazek
# TODO: 4) maska trhlina
# TODO: 5) pocet pixelu a plocha v mm2

## Slovníček pojmů

**Matplotlib**

* **`plt.plot(x, y)`** - čárový graf; každé volání přidá jednu řadu
* **`label` a `legend()`** - text řady a jeho zobrazení v legendě
* **`plt.bar`, `plt.hist`** - sloupcový graf, histogram
* **`fig, ax = plt.subplots()`** - obrázek a osy; kreslí se metodami os,
  `ax.plot(...)`, `ax.set_xlabel(...)`
* **`plt.subplots(2, 1, sharex=True)`** - dva grafy nad sebou se společnou
  osou x
* **`ax.axhline`, `ax.axvline`** - vodorovná, svislá čára přes celý graf
* **`ax.fill_between(x, y1, y2, where=maska)`** - vybarvená plocha mezi
  křivkami
* **`ax.annotate`** - popisek se šipkou
* **`ax.imshow`** - 2D pole nakreslené jako obrázek nebo mapa barev

**NumPy**

* **pole** - `np.array`; všechny prvky mají stejný typ, počítá se prvek
  po prvku
* **`dtype`** - typ prvků pole, např. `float64` nebo `int64`
* **`shape`** - rozměry pole, např. `(120, 200)`
* **`np.linspace(od, do, n)`** - `n` rovnoměrných bodů včetně konce
* **agregace** - metoda, která pole shrne do jednoho čísla: `max`, `mean`,
  `sum`
* **`argmax`** - index největšího prvku
* **maska** - pole `True`/`False` z porovnání, např. `M > M_Rd`
* **`a[maska]`** - výběr prvků, kde maska platí
* **`a[maska] = hodnota`** - přiřazení jen do vybraných prvků
* **`&`, `|`** - spojení masek, každé porovnání v závorce
* **výřez 2D pole** - `a[radky, sloupce]`, např. `a[20:80, 20:100]`

**Náhodná čísla**

* **generátor** - `np.random.default_rng(seed)`; vyrábí náhodná čísla
* **seed** - počáteční číslo generátoru; se stejným seedem vyjdou stejná
  čísla
* **normální rozdělení** - `rng.normal(prumer, odchylka, size=n)`
* **kvantil** - `np.percentile(a, 5)`; hodnota, pod kterou leží 5 % prvků
* **Monte Carlo** - výsledek se spočítá z mnoha nasimulovaných případů

**Interaktivita**

* **`interact(funkce, parametr=(od, do, krok))`** - posuvníky, které při
  změně funkci zavolají znovu